# CDMA Module 4 Targeted Checks on Kaggle

This notebook is focused on **required re-check experiments only**.

It does the following:
1. Clones your GitHub code repository
2. Downloads the features archive from Google Drive
3. Places data into the expected project structure
4. Runs only selected Module 4 conditions across all 5 folds
5. Collects pooled metrics into one summary table

Important: enable **Internet** in Kaggle notebook settings before running.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile
import re

# --------- User configuration ---------
REPO_URL = "https://github.com/ZahirAhmadChaudhry/CDMA_from_Sctrach.git"
REPO_BRANCH = "main"
PROJECT_DIR = Path("/kaggle/working/CDMA_from_Sctrach")

# Keep False to preserve checkpoints/results for resume-safe reruns
RESET_PROJECT_DIR = False

GDRIVE_FILE_ID = "1LJenZ-VXktBbroTI3btVSkRRq-glSWCb"
ARCHIVE_PATH = Path("/kaggle/working/androids_features.zip")
EXTRACT_DIR = Path("/kaggle/working/androids_extracted")

# Run only the experiments that still need investigation
TARGET_CONDITIONS = [
    "ctga_rt",
    "ctga_it",
    "ba4",
    "ba5",
    "full_cdma",
]

# Optional override. Example: ["ctga_rt", "full_cdma"]
CUSTOM_CONDITIONS = None

CONDITIONS = CUSTOM_CONDITIONS if CUSTOM_CONDITIONS else TARGET_CONDITIONS
REPS = [1]  # keep one repetition for targeted checks
EPOCHS = 300

# Independent output folder for this notebook
RESULTS_SUBDIR = "results/module4_targeted_checks"

# Output policy: overwrite keeps one all-folds report/prediction file per mode
OUTPUT_MODE = "overwrite"  # "overwrite" or "detailed"

# Optional one-batch CT-GA diagnostic before targeted sweep
RUN_CTGA_DIAGNOSTIC = True
CTGA_DIAGNOSTIC_MODE = "ctga_rt"
CTGA_DIAGNOSTIC_FOLD = "fold1"
CTGA_DIAGNOSTIC_SOURCE = "train"  # "train" or "test"

print("Configured conditions:", CONDITIONS)
print("Configured reps:", REPS)
print("Configured epochs:", EPOCHS)
print("Configured results subdir:", RESULTS_SUBDIR)
print("Configured output mode:", OUTPUT_MODE)
print("CT-GA diagnostic enabled:", RUN_CTGA_DIAGNOSTIC)
print("RESET_PROJECT_DIR:", RESET_PROJECT_DIR)

In [ ]:
if PROJECT_DIR.exists() and RESET_PROJECT_DIR:
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print("Project already exists. Pulling latest code without deleting local checkpoints/results...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)
    ], check=True)

print("Repository ready at:", PROJECT_DIR)

In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "gdown"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)

print("Dependencies installed.")

In [ ]:
import gdown

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

gdown.download(id=GDRIVE_FILE_ID, output=str(ARCHIVE_PATH), quiet=False)

with zipfile.ZipFile(ARCHIVE_PATH, "r") as zip_file:
    zip_file.extractall(EXTRACT_DIR)

print("Downloaded archive:", ARCHIVE_PATH)
print("Extracted to:", EXTRACT_DIR)

In [ ]:
def find_first(root: Path, name: str):
    matches = [path for path in root.rglob(name)]
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {root}")
    return matches[0]

def find_dir(root: Path, dir_name: str):
    matches = [path for path in root.rglob(dir_name) if path.is_dir()]
    if not matches:
        raise FileNotFoundError(f"Could not find directory {dir_name} under {root}")
    return matches[0]

fold_csv_source = find_first(EXTRACT_DIR, "fold-lists.csv")
cdma_features_source = find_dir(EXTRACT_DIR, "cdma_features")

target_data_dir = PROJECT_DIR / "data"
target_data_dir.mkdir(parents=True, exist_ok=True)

target_fold_csv = target_data_dir / "fold-lists.csv"
target_cdma_dir = target_data_dir / "cdma_features"

if target_fold_csv.exists():
    target_fold_csv.unlink()
if target_cdma_dir.exists():
    shutil.rmtree(target_cdma_dir)

shutil.copy2(fold_csv_source, target_fold_csv)
shutil.copytree(cdma_features_source, target_cdma_dir)

print("Prepared Kaggle data folder at:", target_data_dir)
print("fold-lists.csv:", target_fold_csv.exists())
print("cdma_features exists:", target_cdma_dir.exists())

In [ ]:
import numpy as np

rt_dir = PROJECT_DIR / "data" / "cdma_features" / "rt"
it_dir = PROJECT_DIR / "data" / "cdma_features" / "it"

rt_files = sorted(rt_dir.glob("*_frames.npy"))
it_files = sorted(it_dir.glob("*_frames.npy"))

print("RT files:", len(rt_files))
print("IT files:", len(it_files))

sample_rt = np.load(rt_dir / "01_CF56_1_frames.npy")
sample_it = np.load(it_dir / "01_CF56_1_frames.npy")
print("Sample RT shape:", sample_rt.shape)
print("Sample IT shape:", sample_it.shape)

In [ ]:
results_dir = PROJECT_DIR / RESULTS_SUBDIR

if OUTPUT_MODE == "overwrite" and results_dir.exists():
    removed_count = 0
    for stale_file in results_dir.glob("*_fold*_predictions.csv"):
        stale_file.unlink()
        removed_count += 1
    for stale_file in results_dir.glob("*_fold*_report.txt"):
        stale_file.unlink()
        removed_count += 1
    print("Removed stale detailed files:", removed_count)

if RUN_CTGA_DIAGNOSTIC:
    diagnostic_command = [
        "python",
        "run_module4.py",
        "--mode", CTGA_DIAGNOSTIC_MODE,
        "--rep", str(REPS[0]),
        "--results-dir", RESULTS_SUBDIR,
        "--debug-ctga-batch",
        "--debug-ctga-only",
        "--debug-fold", CTGA_DIAGNOSTIC_FOLD,
        "--debug-batch-source", CTGA_DIAGNOSTIC_SOURCE,
        "--output-mode", OUTPUT_MODE,
    ]
    print("Running CT-GA diagnostic:", " ".join(diagnostic_command))
    subprocess.run(diagnostic_command, cwd=PROJECT_DIR, check=True)

for rep in REPS:
    for mode in CONDITIONS:
        command = [
            "python",
            "run_module4.py",
            "--mode", mode,
            "--all-folds",
            "--rep", str(rep),
            "--epochs", str(EPOCHS),
            "--results-dir", RESULTS_SUBDIR,
            "--output-mode", OUTPUT_MODE,
            "--log-every-epochs", "50",
            "--checkpoint-every-epochs", "50",
            "--preview-participants", "5",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, cwd=PROJECT_DIR, check=True)

print("All targeted Module 4 runs completed.")

In [ ]:
import pandas as pd

results_dir = PROJECT_DIR / RESULTS_SUBDIR
rows = []

for rep in REPS:
    for mode in CONDITIONS:
        report_path = results_dir / f"{mode}_rep{rep}_all_folds_report.txt"
        if not report_path.exists():
            print("Missing report:", report_path)
            continue

        metrics = {}
        for line in report_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line.startswith("- accuracy:"):
                metrics["accuracy"] = float(line.split(":", 1)[1].strip())
            elif line.startswith("- precision:"):
                metrics["precision"] = float(line.split(":", 1)[1].strip())
            elif line.startswith("- recall:"):
                metrics["recall"] = float(line.split(":", 1)[1].strip())
            elif line.startswith("- f1:"):
                metrics["f1"] = float(line.split(":", 1)[1].strip())

        rows.append({
            "rep": rep,
            "mode": mode,
            "accuracy": metrics.get("accuracy"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "f1": metrics.get("f1"),
            "report_path": str(report_path),
        })

summary_df = pd.DataFrame(rows).sort_values(["rep", "mode"]).reset_index(drop=True)
display(summary_df)

summary_csv = results_dir / "module4_targeted_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved summary CSV:", summary_csv)

## Notes
- This notebook runs **targeted conditions only** by default: `ctga_rt`, `ctga_it`, `ba4`, `ba5`, `full_cdma`.
- Use `CUSTOM_CONDITIONS` if you want a smaller custom subset.
- Outputs are isolated under `results/module4_targeted_checks` so this run is independent of prior full-13 runs.
- `OUTPUT_MODE = "overwrite"` avoids per-fold result-file growth and continuously updates one all-folds report/predictions file per mode.
- Optional CT-GA one-batch diagnostic runs once before training and writes a diagnostic report file.
- Default setup runs 1 repetition and full 300 epochs.
- Keep `RESET_PROJECT_DIR = False` to preserve checkpoints/results between reruns.
- Terminal logs show loss at epoch 1, every 50 epochs, and final epoch.
- Terminal logs print the first 5 participant predictions after each fold.
- Checkpoints are saved every 50 epochs and at final epoch under `results/module4_targeted_checks/checkpoints`.
- Resume is enabled by default: reruns skip completed folds and continue from latest checkpoint when available.
- Summary CSV is saved at `results/module4_targeted_checks/module4_targeted_summary.csv`.